# NovaCart — Silver Order Items Transformation

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 1. Define Storage Paths

In [0]:
BRONZE_ORDER_ITEMS_PATH = (
    "abfss://bronze@stnovacartdev.dfs.core.windows.net/"
    "olist/order_items"
)

SILVER_ORDER_ITEMS_PATH = (
    "abfss://silver@stnovacartdev.dfs.core.windows.net/"
    "olist/order_items"
)

QUARANTINE_ORDER_ITEMS_PATH = (
    "abfss://quarantine@stnovacartdev.dfs.core.windows.net/"
    "olist/order_items"
)

print(f"Bronze path: {BRONZE_ORDER_ITEMS_PATH}")
print(f"Silver path: {SILVER_ORDER_ITEMS_PATH}")
print(f"Quarantine path: {QUARANTINE_ORDER_ITEMS_PATH}")

## 2. Read Bronze Order Items Data

In [0]:
order_items_bronze_df = (
    spark.read
    .format("delta")
    .load(BRONZE_ORDER_ITEMS_PATH)
)

bronze_row_count = order_items_bronze_df.count()

print("Bronze order items loaded successfully.")
print(f"Bronze row count: {bronze_row_count}")

order_items_bronze_df.printSchema()
display(order_items_bronze_df.limit(10))

## 3. Validate Required Columns

In [0]:
required_columns = [
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "shipping_limit_date",
    "price",
    "freight_value",
    "_source_file",
    "_ingestion_timestamp",
    "_batch_id",
]

missing_columns = [
    column
    for column in required_columns
    if column not in order_items_bronze_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required Bronze columns: {missing_columns}"
    )

print("Required-column validation passed.")

## 4. Profile Missing and Invalid Values

In [0]:
order_items_profile_df = order_items_bronze_df.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        (
            F.col("order_id").isNull()
            | (F.trim(F.col("order_id")) == "")
        ).cast("int")
    ).alias("invalid_order_id"),

    F.sum(
    (
        F.col("order_item_id").isNull()
        | (F.col("order_item_id") <= 0)
    ).cast("int")
).alias("invalid_order_item_id"),

    F.sum(
        (
            F.col("product_id").isNull()
            | (F.trim(F.col("product_id")) == "")
        ).cast("int")
    ).alias("invalid_product_id"),

    F.sum(
        (
            F.col("seller_id").isNull()
            | (F.trim(F.col("seller_id")) == "")
        ).cast("int")
    ).alias("invalid_seller_id"),

    F.sum(
        F.col("shipping_limit_date").isNull().cast("int")
    ).alias("missing_shipping_limit_date"),

    F.sum(
        F.col("price").isNull().cast("int")
    ).alias("missing_price"),

    F.sum(
        (F.col("price") < 0).cast("int")
    ).alias("negative_price"),

    F.sum(
        F.col("freight_value").isNull().cast("int")
    ).alias("missing_freight_value"),

    F.sum(
        (F.col("freight_value") < 0).cast("int")
    ).alias("negative_freight_value"),
)

display(order_items_profile_df)

## 5. Check Duplicate Order Item Keys

In [0]:
duplicate_order_item_keys_df = (
    order_items_bronze_df
    .groupBy(
        "order_id",
        "order_item_id"
    )
    .count()
    .filter(
        F.col("order_id").isNotNull()
        & F.col("order_item_id").isNotNull()
        & (F.col("count") > 1)
    )
)

duplicate_order_item_key_count = (
    duplicate_order_item_keys_df.count()
)

print(
    "Number of duplicate "
    "(order_id, order_item_id) keys: "
    f"{duplicate_order_item_key_count}"
)

display(duplicate_order_item_keys_df.limit(20))

## 6. Check Exact Duplicate Records

In [0]:
business_columns = [
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "shipping_limit_date",
    "price",
    "freight_value",
]

exact_duplicate_count = (
    bronze_row_count
    - order_items_bronze_df
        .dropDuplicates(business_columns)
        .count()
)

print(f"Exact duplicate order item rows: {exact_duplicate_count}")

## 7. Clean and Standardize Order Item Fields

In [0]:
order_items_cleaned_df = (
    order_items_bronze_df
    .withColumn(
        "order_id",
        F.trim(F.col("order_id"))
    )
    .withColumn(
        "product_id",
        F.trim(F.col("product_id"))
    )
    .withColumn(
        "seller_id",
        F.trim(F.col("seller_id"))
    )
)

## 8. Detect Duplicate Composite Keys

In [0]:
from pyspark.sql.window import Window

order_item_key_window = Window.partitionBy(
    "order_id",
    "order_item_id"
)

order_items_checked_df = (
    order_items_cleaned_df
    .withColumn(
        "_duplicate_key_count",
        F.count("*").over(order_item_key_window)
    )
)

## 9. Define Order Item Validation Rules

In [0]:
invalid_order_id_condition = (
    F.col("order_id").isNull()
    | (F.col("order_id") == "")
)

invalid_order_item_id_condition = (
    F.col("order_item_id").isNull()
    | (F.col("order_item_id") <= 0)
)

invalid_product_id_condition = (
    F.col("product_id").isNull()
    | (F.col("product_id") == "")
)

invalid_seller_id_condition = (
    F.col("seller_id").isNull()
    | (F.col("seller_id") == "")
)

invalid_shipping_limit_date_condition = (
    F.col("shipping_limit_date").isNull()
)

invalid_price_condition = (
    F.col("price").isNull()
    | (F.col("price") < 0)
)

invalid_freight_value_condition = (
    F.col("freight_value").isNull()
    | (F.col("freight_value") < 0)
)

duplicate_order_item_key_condition = (
    F.col("_duplicate_key_count") > 1
)

## 10. Assign Order Item Rejection Reasons

In [0]:
order_items_validated_df = order_items_checked_df.withColumn(
    "_rejection_reason",

    F.when(
        invalid_order_id_condition,
        F.lit("MISSING_ORDER_ID")
    )
    .when(
        invalid_order_item_id_condition,
        F.lit("INVALID_ORDER_ITEM_ID")
    )
    .when(
        invalid_product_id_condition,
        F.lit("MISSING_PRODUCT_ID")
    )
    .when(
        invalid_seller_id_condition,
        F.lit("MISSING_SELLER_ID")
    )
    .when(
        invalid_shipping_limit_date_condition,
        F.lit("MISSING_SHIPPING_LIMIT_DATE")
    )
    .when(
        invalid_price_condition,
        F.lit("INVALID_PRICE")
    )
    .when(
        invalid_freight_value_condition,
        F.lit("INVALID_FREIGHT_VALUE")
    )
    .when(
        duplicate_order_item_key_condition,
        F.lit("DUPLICATE_ORDER_ITEM_KEY")
    )
    .otherwise(F.lit(None))
)

## 11. Review Validation Results

In [0]:
display(
    order_items_validated_df
    .groupBy("_rejection_reason")
    .count()
    .orderBy("_rejection_reason")
)

## 12. Split Valid and Invalid Order Items

In [0]:
order_items_valid_df = (
    order_items_validated_df
    .filter(F.col("_rejection_reason").isNull())
    .drop(
        "_rejection_reason",
        "_duplicate_key_count"
    )
)

order_items_quarantine_df = (
    order_items_validated_df
    .filter(F.col("_rejection_reason").isNotNull())
    .drop("_duplicate_key_count")
)

## 13. Add Silver Processing Metadata

In [0]:
order_items_silver_df = (
    order_items_valid_df
    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

## 14. Add Quarantine Metadata

In [0]:
order_items_quarantine_df = (
    order_items_quarantine_df
    .withColumn(
        "_quarantined_at",
        F.current_timestamp()
    )
    .withColumn(
        "_source_dataset",
        F.lit("order_items")
    )
)

## 15. Count Silver and Quarantine Records

In [0]:
valid_row_count = order_items_silver_df.count()
quarantine_row_count = order_items_quarantine_df.count()

print(f"Valid Silver rows: {valid_row_count}")
print(f"Quarantined rows: {quarantine_row_count}")
print(f"Bronze input rows: {bronze_row_count}")

## 16. Validate Row-Count Reconciliation

In [0]:
if valid_row_count + quarantine_row_count != bronze_row_count:
    raise ValueError(
        "Row-count validation failed: "
        "Silver rows + quarantine rows do not equal Bronze input rows."
    )

print("Row-count validation passed.")

## 17. Write Valid Order Items to Silver

In [0]:
(
    order_items_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_ORDER_ITEMS_PATH)
)

print("Silver order items written successfully.")

## 18. Write Invalid Order Items to Quarantine

In [0]:
(
    order_items_quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(QUARANTINE_ORDER_ITEMS_PATH)
)

print("Order items quarantine output written successfully.")

## 19. Read Written Delta Outputs

In [0]:
order_items_silver_written_df = (
    spark.read
    .format("delta")
    .load(SILVER_ORDER_ITEMS_PATH)
)

order_items_quarantine_written_df = (
    spark.read
    .format("delta")
    .load(QUARANTINE_ORDER_ITEMS_PATH)
)

silver_written_count = order_items_silver_written_df.count()
quarantine_written_count = order_items_quarantine_written_df.count()

print(f"Written Silver rows: {silver_written_count}")
print(f"Written quarantine rows: {quarantine_written_count}")

## 20. Validate Written Outputs

In [0]:
if silver_written_count != valid_row_count:
    raise ValueError(
        "Silver write validation failed: "
        f"expected {valid_row_count}, wrote {silver_written_count}."
    )

if quarantine_written_count != quarantine_row_count:
    raise ValueError(
        "Quarantine write validation failed: "
        f"expected {quarantine_row_count}, wrote "
        f"{quarantine_written_count}."
    )

if silver_written_count + quarantine_written_count != bronze_row_count:
    raise ValueError(
        "Final reconciliation failed: "
        "Silver + quarantine does not equal Bronze."
    )

print("Silver order items pipeline completed successfully.")
print("Final row-count validation passed.")

## 21. Inspect Final Silver Order Items Dataset

In [0]:
order_items_silver_written_df.printSchema()

display(
    order_items_silver_written_df.select(
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "shipping_limit_date",
        "price",
        "freight_value",
        "_silver_processed_at"
    ).limit(20)
)